In [0]:
display(dbutils.fs.ls("/Volumes/pharmacy_sales/bronze/bronze_layer/bronze_vw_sales.parquet"))

path,name,size,modificationTime
dbfs:/Volumes/pharmacy_sales/bronze/bronze_layer/bronze_vw_sales.parquet,bronze_vw_sales.parquet,172962964,1788979436000


In [0]:
df = spark.read.parquet("/Volumes/pharmacy_sales/bronze/bronze_layer/bronze_vw_sales.parquet")

In [0]:
# Use PyArrow to handle nanosecond timestamp, then convert to microseconds
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# Read with PyArrow
table = pq.read_table("/Volumes/pharmacy_sales/bronze/bronze_layer/bronze_vw_sales.parquet")

# Convert nanosecond timestamp to microsecond timestamp (Spark-compatible)
if 'BILLDATETIME' in table.column_names:
    billdatetime_col = table.column('BILLDATETIME')
    # Cast from nanosecond to microsecond precision
    billdatetime_micro = pc.cast(billdatetime_col, pa.timestamp('us'))
    # Replace the column
    table = table.set_column(table.schema.get_field_index('BILLDATETIME'), 'BILLDATETIME', billdatetime_micro)

# Convert to Spark DataFrame
df = spark.createDataFrame(table.to_pandas())
print("Rows: ",df.count())
display(df.limit(20))

Rows:  7176819


STORE,custgroup,itemid,itemname,ctg,subctg,packsize,Generic_Flag,BILLNO,BILLDATETIME,SALEQTY,BILLDATE,SALEVAL,DISCOUNT,BILLS,MAXSALEQTY,OFFERFLAG
18896,0,EVO0067,EVOCUS H20 ALKALINE WATER 500ML,FMCG,FOOD & BEVERAGES,1,NULL,CS0000600,2024-02-04T13:52:00.000Z,1,2024-02-04,100.0,0.0,1,1,NULL
18896,0,ENO0007,ENO LEMON 100G,FMCG,OTC,1,NULL,CS0005672,2025-03-14T14:37:00.000Z,1,2025-03-14,185.0,0.0,1,1,NULL
18896,0,HIM0251,HIMALAYA OIL CLEAR LEMON FACE WASH 50 ML,FMCG,PERSONAL CARE,1,NULL,CS0011286,2026-03-18T07:24:00.000Z,1,2026-03-18,95.0,0.0,1,1,NULL
18312,5020,WYS0003,WYSOLONE DT 5MG TAB 15'S,PHARMA,TABLET,15,NULL,CS0171004,2026-05-18T21:47:00.000Z,15,2026-05-18,10.800000190734863,1.0800000429153442,1,15,NULL
18693,0,FEB0235,FEBUTAZ 40 TAB 15'S,PHARMA,TABLET,15,NULL,SI00000113,2026-06-26T08:38:00.000Z,15,2026-06-26,252.89999389648438,0.0,1,15,NULL
18693,0,ABZ0016,ABZORB DUSTING POWDER 120G,PHARMA,POWDER,1,NULL,WS0019402,2025-10-06T13:57:00.000Z,1,2025-10-06,164.05999755859375,0.0,1,1,NULL
18693,7661,APL0118,AP LIFE COCONUT JAGGERY POWDER 250G,PRIVATE LABEL,PRIVATE LABEL,1,NULL,WS0009410,2024-11-27T19:30:00.000Z,1,2024-11-27,249.0,0.0,1,1,NULL
18312,7661,NEP0210,NEPTAZ 100MG TAB 14'S,PHARMA,TABLET,14,NULL,CS0163271,2026-03-07T15:51:00.000Z,14,2026-03-07,503.8599853515625,0.0,1,14,NULL
18693,102,VIL0176,VILDANIZ-MF 50/500MG TAB 15'S,PHARMA,TABLET,15,NULL,CC0002409,2024-01-17T12:05:00.000Z,15,2024-01-17,195.0,0.0,1,15,NULL
18693,102,SEL0012,SELOKEN XL 25MG TAB 30'S,PHARMA,TABLET,30,NULL,CS0024998,2024-04-26T08:06:00.000Z,30,2024-04-26,141.0,0.0,1,30,NULL


In [0]:
# Verify the Schema
df.printSchema()

root
 |-- STORE: long (nullable = true)
 |-- custgroup: long (nullable = true)
 |-- itemid: string (nullable = true)
 |-- itemname: string (nullable = true)
 |-- ctg: string (nullable = true)
 |-- subctg: string (nullable = true)
 |-- packsize: long (nullable = true)
 |-- Generic_Flag: string (nullable = true)
 |-- BILLNO: string (nullable = true)
 |-- BILLDATETIME: timestamp (nullable = true)
 |-- SALEQTY: long (nullable = true)
 |-- BILLDATE: date (nullable = true)
 |-- SALEVAL: double (nullable = true)
 |-- DISCOUNT: double (nullable = true)
 |-- BILLS: long (nullable = true)
 |-- MAXSALEQTY: long (nullable = true)
 |-- OFFERFLAG: string (nullable = true)



In [0]:
df.write.mode("overwrite").saveAsTable("bronze_vw_sales")